# Rock pick auto-labeling with CoTracker3 on Google Colab

Place the 5 keypoints on the rockpick frame presented, then CoTracker3 will track them through each frame.

**Output image settings (Cell 2)**
- Upload videos at their recorded resolution. Saved images are always 640×480.
- `CENTER_ON_KEYPOINTS = False` (default) crops around the image center.
- `CENTER_ON_KEYPOINTS = True` crops around the visible keypoint bounding-box center.
- Both modes preserve source pixel scale. Smaller dimensions receive black padding.
- Click points on the original frame. Exported coordinates are translated into the crop.
- Keep `CENTER_ON_KEYPOINTS` fixed while collecting a dataset. Each mode has its own output folder.
- Inspect the crops for truncated targets. Cell 6 removes frames with fewer than five visible points.

**How to use**
1. Run cells **1–2** once per session.
2. For **each clip**: run all cells in sequential order **except for the last one** for each video clip
3. Run the final cell to download the completed dataset

!!! Remember to download when finished if using Google Colab or files will not save !!!

In [ ]:
# 1) check GPU and load CoTracker3
import os, json, base64, shutil, math, subprocess, glob
import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import display, Javascript
from google.colab import files
from google.colab.output import eval_js

assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then rerun this cell (if using Colab)"
device = "cuda"
model = torch.hub.load("facebookresearch/co-tracker", "cotracker3_offline").to(device).eval()
print("CoTracker3 loaded on", torch.cuda.get_device_name(0))

In [ ]:
# 2) Settings + helper functions

# Must match Roboflow's keypoint order (checked against the COCO export)
KEYPOINTS = ["blunt-end", "metal-intersection", "sharp-end", "metal-handle-intersection", "handle"]
CLASS_ID, CLASS_NAME = 0, "rock_pick"

EVERY = 16          # save every 8th frame (30 fps -> ~4 labeled images per second of video)
PAD = 0.08         # bounding-box padding around the keypoints (fraction of box size)
MIN_VISIBLE = 3    # skip a frame if fewer than this many points are visible
MAX_SIDE = 640     # downscale for tracking only (saves GPU memory); labels are full-res
TRACK_STRIDE = 8   # track every 2nd frame to fit in GPU memory (use 4 if you still get "out of memory")
OUTPUT_WIDTH, OUTPUT_HEIGHT = 640, 480
CENTER_ON_KEYPOINTS = False  # True follows the visible keypoints; False uses the image center.
# Separate the two modes so changing this setting does not mix generated images.
OUT = "out_640x480_keypoints" if CENTER_ON_KEYPOINTS else "out_640x480_center"

COLORS_BGR = [(0,140,255), (255,0,255), (255,229,0), (0,255,0), (48,48,255)]

def read_frame(video, idx):
    cap = cv2.VideoCapture(video); cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read(); cap.release()
    if not ok: raise RuntimeError(f"Could not read frame {idx}")
    return frame

def crop_output_frame(
    frame: np.ndarray,
    kp: np.ndarray,
    vis: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Crop/pad to 640x480 at source pixel scale and translate keypoints."""
    frame_h, frame_w = frame.shape[:2]
    source_w = min(OUTPUT_WIDTH, frame_w)
    source_h = min(OUTPUT_HEIGHT, frame_h)
    valid = (
        np.asarray(vis, dtype=bool)
        & np.isfinite(kp).all(axis=1)
        & (kp[:, 0] >= 0) & (kp[:, 0] < frame_w)
        & (kp[:, 1] >= 0) & (kp[:, 1] < frame_h)
    )

    center_x, center_y = frame_w / 2, frame_h / 2
    if CENTER_ON_KEYPOINTS and valid.any():
        points = kp[valid]
        center_x = float(points[:, 0].min() + points[:, 0].max()) / 2
        center_y = float(points[:, 1].min() + points[:, 1].max()) / 2

    x0 = int(np.clip(round(center_x - source_w / 2), 0, frame_w - source_w))
    y0 = int(np.clip(round(center_y - source_h / 2), 0, frame_h - source_h))
    cropped = frame[y0:y0 + source_h, x0:x0 + source_w]
    pad_left = (OUTPUT_WIDTH - source_w) // 2
    pad_top = (OUTPUT_HEIGHT - source_h) // 2
    cropped = cv2.copyMakeBorder(
        cropped,
        pad_top, OUTPUT_HEIGHT - source_h - pad_top,
        pad_left, OUTPUT_WIDTH - source_w - pad_left,
        cv2.BORDER_CONSTANT, value=(0, 0, 0),
    )

    translated = np.asarray(kp, dtype=float).copy()
    translated[:, 0] += pad_left - x0
    translated[:, 1] += pad_top - y0
    visible = (
        valid
        & (translated[:, 0] >= 0) & (translated[:, 0] < OUTPUT_WIDTH)
        & (translated[:, 1] >= 0) & (translated[:, 1] < OUTPUT_HEIGHT)
    )
    return cropped, translated, visible


CLICK_JS = '''
window.clickPoints = async function(b64, names) {
  const colors = ['#ff8c00','#ff00ff','#00e5ff','#00ff00','#ff3030'];
  const wrap = document.createElement('div');
  const msg = document.createElement('div');
  msg.style.cssText = 'font:16px sans-serif;margin:6px 0;padding:6px 10px;background:#222;color:#fff';
  const canvas = document.createElement('canvas');
  canvas.style.cssText = 'max-width:100%;cursor:crosshair;display:block';
  const bar = document.createElement('div'); bar.style.margin = '8px 0';
  const undo = document.createElement('button'); undo.textContent = 'Undo';
  const done = document.createElement('button'); done.textContent = 'Done'; done.style.marginLeft = '8px';
  bar.append(undo, done); wrap.append(msg, canvas, bar); document.body.appendChild(wrap);
  const img = new Image();
  await new Promise(r => { img.onload = r; img.src = 'data:image/jpeg;base64,' + b64; });
  canvas.width = img.width; canvas.height = img.height;
  const ctx = canvas.getContext('2d'); const pts = [];
  function draw() {
    ctx.drawImage(img, 0, 0);
    pts.forEach((p, i) => {
      ctx.fillStyle = colors[i]; ctx.beginPath(); ctx.arc(p[0], p[1], 7, 0, 2*Math.PI); ctx.fill();
      ctx.font = '20px sans-serif'; ctx.lineWidth = 4; ctx.strokeStyle = '#000';
      ctx.strokeText(names[i], p[0]+10, p[1]-10); ctx.fillText(names[i], p[0]+10, p[1]-10);
    });
    msg.textContent = pts.length < names.length
      ? `Click ${pts.length+1}/${names.length}: ${names[pts.length]}`
      : 'All points placed - click Done (or Undo to fix one)';
    done.disabled = pts.length !== names.length;
  }
  canvas.onclick = e => {
    if (pts.length >= names.length) return;
    const r = canvas.getBoundingClientRect();
    pts.push([Math.round((e.clientX - r.left) * canvas.width / r.width),
              Math.round((e.clientY - r.top) * canvas.height / r.height)]);
    draw();
  };
  undo.onclick = () => { pts.pop(); draw(); };
  draw();
  await new Promise(r => done.onclick = r);
  bar.remove(); msg.textContent = 'Saved: ' + JSON.stringify(pts);
  return pts;
}
'''

def click_points(frame, names):
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
    b64 = base64.b64encode(buf).decode()
    display(Javascript(CLICK_JS))
    return eval_js(f"clickPoints('{b64}', {json.dumps(names)})")

def yolo_line(kp: np.ndarray, vis: np.ndarray, W: int, H: int, pad: float) -> str:
    valid = (
        np.asarray(vis, dtype=bool) & np.isfinite(kp).all(axis=1)
        & (kp[:, 0] >= 0) & (kp[:, 0] < W)
        & (kp[:, 1] >= 0) & (kp[:, 1] < H)
    )
    if not valid.any():
        raise ValueError("Cannot export a box without visible keypoints")
    xs, ys = kp[valid, 0], kp[valid, 1]
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    p = pad * max(x1 - x0, y1 - y0)
    x0, y0 = max(0, x0 - p), max(0, y0 - p)
    x1, y1 = min(W - 1, x1 + p), min(H - 1, y1 + p)
    box = ((x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H)
    parts = [str(CLASS_ID)] + [f"{value:.6f}" for value in box]
    for (x, y), visible in zip(kp, valid):
        if visible:
            parts += [f"{x / W:.6f}", f"{y / H:.6f}", "2"]
        else:
            parts += ["0", "0", "0"]
    return " ".join(parts)

def write_data_yaml():
    os.makedirs(OUT, exist_ok=True)
    with open(os.path.join(OUT, "data.yaml"), "w") as f:
        f.write("path: .\ntrain: train/images\nval: train/images\n"
                f"kpt_shape: [{len(KEYPOINTS)}, 3]\nflip_idx: {list(range(len(KEYPOINTS)))}\n"
                f"names:\n  {CLASS_ID}: {CLASS_NAME}\n# keypoint order: {KEYPOINTS}\n")

def track_and_save(video, points, click_frame):
    assert EVERY % TRACK_STRIDE == 0, "EVERY must be a multiple of TRACK_STRIDE"
    assert click_frame % TRACK_STRIDE == 0, f"CLICK_FRAME must be a multiple of TRACK_STRIDE ({TRACK_STRIDE})"
    # load every TRACK_STRIDE-th frame, downscaled, for tracking
    cap = cv2.VideoCapture(video); small = []; W = H = None; s = 1.0; T = 0
    while True:
        ok, f = cap.read()
        if not ok: break
        if W is None:
            H, W = f.shape[:2]; s = min(1.0, MAX_SIDE / max(H, W))
        if T % TRACK_STRIDE == 0:
            if s < 1.0: f = cv2.resize(f, (round(W*s), round(H*s)), interpolation=cv2.INTER_AREA)
            small.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        T += 1
    cap.release()
    print(f"{T} frames at {W}x{H}; tracking {len(small)} of them (every {TRACK_STRIDE})...")

    if not small:
        raise ValueError(f"No readable video frames: {video}")
    tracking_h, tracking_w = small[0].shape[:2]
    scale_xy = np.array([tracking_w / W, tracking_h / H])
    torch.cuda.empty_cache()
    vid = torch.from_numpy(np.stack(small)).permute(0, 3, 1, 2)[None].float().to(device)
    qf = click_frame // TRACK_STRIDE
    q = torch.tensor([[[qf, x*scale_xy[0], y*scale_xy[1]] for x, y in points]], dtype=torch.float32, device=device)
    with torch.no_grad():
        tracks, visible = model(vid, queries=q, backward_tracking=qf > 0)
    del vid, q; torch.cuda.empty_cache()
    tracks = tracks[0].cpu().numpy() / scale_xy          # [T/stride, N, 2]
    visible = visible[0].cpu().numpy() > 0.5
    visible &= (tracks[..., 0] >= 0) & (tracks[..., 0] < W) & (tracks[..., 1] >= 0) & (tracks[..., 1] < H)

    clip = os.path.splitext(os.path.basename(video))[0]
    img_dir, lbl_dir = os.path.join(OUT, "train", "images"), os.path.join(OUT, "train", "labels")
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True); write_data_yaml()

    saved, skipped, previews = [], 0, []
    cap = cv2.VideoCapture(video)
    for i in range(T):
        ok, frame = cap.read()
        if not ok: break
        if i % EVERY: continue
        ti = i // TRACK_STRIDE
        if ti >= len(tracks): break
        kp, vis = tracks[ti], visible[ti]
        frame, kp, vis = crop_output_frame(frame, kp, vis)
        if vis.sum() < MIN_VISIBLE:
            skipped += 1; continue
        name = f"{clip}_f{i:05d}"
        if not cv2.imwrite(os.path.join(img_dir, name + ".jpg"), frame):
            raise OSError(f"Could not write output image: {name}")
        with open(os.path.join(lbl_dir, name + ".txt"), "w") as f:
            f.write(yolo_line(kp, vis, OUTPUT_WIDTH, OUTPUT_HEIGHT, PAD) + "\n")
        saved.append(i)
        disp = frame.copy()
        for j, ((x, y), v) in enumerate(zip(kp, vis)):
            if not np.isfinite([x, y]).all() or not (0 <= x < OUTPUT_WIDTH and 0 <= y < OUTPUT_HEIGHT):
                continue
            cv2.circle(disp, (int(x), int(y)), 8, COLORS_BGR[j] if v else (0, 0, 255), -1 if v else 2)
        previews.append((i, cv2.resize(disp, (OUTPUT_WIDTH // 2, OUTPUT_HEIGHT // 2))))
    cap.release()
    print(f"Saved {len(saved)} labeled frames from {clip}, skipped {skipped} (too few visible points).")
    return clip, previews

def show_previews(clip, previews, cols=4):
    if not previews:
        print(f"{clip}: no frames retained; inspect crop mode and visible keypoints.")
        return
    rows = math.ceil(len(previews) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 2.6 * rows))
    for ax in np.array(axes).flat: ax.axis("off")
    for ax, (i, img) in zip(np.array(axes).flat, previews):
        ax.imshow(img[..., ::-1]); ax.set_title(f"frame {i}", fontsize=10)
    plt.suptitle(f"{clip}: filled dot = visible, red ring = hidden", y=1.0); plt.tight_layout(); plt.show()

print(f"Ready. Output: {OUTPUT_WIDTH}x{OUTPUT_HEIGHT}; "
      f"center on {'keypoints' if CENTER_ON_KEYPOINTS else 'image'}; folder: {OUT}")

In [ ]:
# 3) Upload one clip (.mkv, .mp4 or .mov)
os.makedirs("clips", exist_ok=True)
uploaded_file = files.upload()
name = list(uploaded_file)[-1]

# Convert if .mkv (common when recording with guvcview on ubuntu)
if name.lower().endswith(".mkv"):
    converted_to_mp4 = os.path.splitext(name)[0] + ".mp4"
    subprocess.run(["ffmpeg", "-i", name, "-c", "copy", converted_to_mp4, "-y"], check=True)
    name = converted_to_mp4
VIDEO = os.path.join("clips", name)
shutil.move(name, VIDEO)

cap = cv2.VideoCapture(VIDEO)
n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"{VIDEO}: {n} frames, {fps:.0f} fps, {n/max(fps,1):.1f} s, {w}x{h}")

In [ ]:
# 4) Click the 5 points (in the order shown above the image), then press "Done".
# If frame 0 is blurry or a point is hidden, change CLICK_FRAME (e.g. 30) and rerun.
CLICK_FRAME = 0
POINTS = click_points(read_frame(VIDEO, CLICK_FRAME), KEYPOINTS)
print(dict(zip(KEYPOINTS, POINTS)))

In [ ]:
# 5) Track with CoTracker3 and write YOLO labels (~1 min for a 15 s clip)
# If you get "CUDA out of memory": Runtime > Restart session, set TRACK_STRIDE = 4 in cell 2,
# then rerun cells 1-5 (clips longer than ~20 s may also need trimming).
CLIP, PREVIEWS = track_and_save(VIDEO, POINTS, CLICK_FRAME)
show_previews(CLIP, PREVIEWS)

In [ ]:
# 6) Delete bad frames: look at the grid above. If dots slid off the pick on a frame,
# put its frame number in BAD and run this cell. Leave BAD empty if all look good.

# TODO add system to throw out images without 5 keypoints present
BAD = []   # e.g. [96, 104]

# Append images with less than 5 keypoints to BAD
for p in glob.glob(os.path.join(OUT, "train", "labels", f"{CLIP}_f*.txt")):
    vis = open(p).read().split()[7::3]
    if sum(float(v) > 0 for v in vis) < 5:
        BAD.append(int(p[-9:-4]))

for i in BAD:
    for sub, ext in (("images", ".jpg"), ("labels", ".txt")):
        p = os.path.join(OUT, "train", sub, f"{CLIP}_f{i:05d}{ext}")
        if os.path.exists(p): os.remove(p)
total = len(os.listdir(os.path.join(OUT, "train", "images")))
print(f"Removed {len(BAD)} frames. Total labeled images so far (all clips): {total}")
print("Next clip? Go back to cell 3. All done? Run cell 7.")

In [ ]:
# 7) Download everything as one zip -> upload it in Roboflow (Upload Data)
shutil.make_archive("rock_pick_labels", "zip", OUT)
files.download("rock_pick_labels.zip")